# 16a — MP-Declare Constraint Mining (RuM) — DomesticDeclarations

Mines MP-Declare constraints with data conditions from the DomesticDeclarations event log using RuM's MINERful + MpEnhancer.
Categorizes into prefix-safe vs sequence constraints and saves to pkl for downstream notebooks.

In [ ]:
import sys
import os
from pathlib import Path

os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home'

_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir():
        break
    _current = _current.parent

if str(_current) not in sys.path:
    sys.path.insert(0, str(_current))

# src/ must be on sys.path for torch.load to unpickle event_log_loader classes
src_path = str(_current / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from src.interpretability.perturbation_methods import csv_to_xes, discover_mpdeclare
from src.interpretability.perturbation_methods import mpdeclare_to_declare_constraints
from src.interpretability.perturbation_methods import DeclareConstraintChecker

In [ ]:
# ===== CONFIGURATION =====

# Declare mining hyperparameters
MIN_SUPPORT = 0.95
DATA_CONDITIONS = 'ACTIVATIONS'

In [ ]:
csv_path = _current / 'data' / 'domestic_declarations.csv'
xes_path = _current / 'data' / 'DomesticDeclarations.xes'

if not xes_path.exists():
    csv_to_xes(csv_path, xes_path, case_id_col='Case ID', activity_col='Activity', timestamp_col='Complete Timestamp')
    print(f'Converted CSV to XES: {xes_path}')
else:
    print(f'XES file already exists: {xes_path}')

In [ ]:
# Start JVM with extra heap before discover_mpdeclare
import jpype
if not jpype.isJVMStarted():
    from src.interpretability.perturbation_methods.revised_plus.rum_mpdeclare import RUM_JAR
    jpype.startJVM(f'-Djava.class.path={RUM_JAR}', '-Djava.awt.headless=true', '-Xmx4g', convertStrings=True)
    __import__('jpype.imports')

mpdeclare_constraints = discover_mpdeclare(
    xes_path,
    min_support=MIN_SUPPORT,
    data_conditions=DATA_CONDITIONS,
)
print(f'Mined {len(mpdeclare_constraints)} MP-Declare constraints')

In [ ]:
# Build activity vocabulary from dataset
import torch
from tqdm.auto import tqdm
from src.interpretability.utils.tensor_decoder import TensorDecoder

with tqdm(total=2, desc='Loading dataset') as pbar:
    pbar.set_postfix_str('loading pkl...')
    data_path = _current / 'encoded_data' / 'test_philipp' / 'domestic_declarations_all_5_test.pkl'
    full_dataset = torch.load(data_path, weights_only=False)
    pbar.update(1)

    pbar.set_postfix_str('building vocabulary...')
    decoder = TensorDecoder(full_dataset)

    ACTIVITY_FEATURE = 'Activity'
    activity_idx_to_label = decoder.idx_to_label[ACTIVITY_FEATURE]
    max_idx = max(activity_idx_to_label.keys())
    activity_names = [activity_idx_to_label.get(i, f'<unk_{i}>') for i in range(max_idx + 1)]

    # Build name -> index mapping for conversion
    activity_name_to_idx = {name: i for i, name in enumerate(activity_names)}
    pbar.update(1)
    pbar.set_postfix_str('done')

print(f'Activity vocabulary ({len(activity_names)}):')
for i, name in enumerate(activity_names):
    print(f'  {i}: {name}')

In [ ]:
# Convert MPDeclareConstraint -> DeclareConstraint
from tqdm.auto import tqdm

with tqdm(total=3, desc='Post-processing constraints') as pbar:
    pbar.set_postfix_str('converting to DeclareConstraint...')
    all_constraints, data_conditions = mpdeclare_to_declare_constraints(
        mpdeclare_constraints, activity_name_to_idx
    )
    pbar.update(1)

    pbar.set_postfix_str('categorizing...')
    prefix_safe_constraints = set(DeclareConstraintChecker.prefix_safe_constraints(all_constraints))
    sequence_constraints = all_constraints - prefix_safe_constraints
    pbar.update(1)

    pbar.set_postfix_str('done')
    pbar.update(1)

print(f'Total constraints:        {len(all_constraints)}')
print(f'Prefix-safe constraints:  {len(prefix_safe_constraints)}')
print(f'Sequence constraints:     {len(sequence_constraints)}')
print(f'Data conditions:          {len(data_conditions)}')

In [ ]:
# Display all three categories
print('=' * 80)
print('PREFIX-SAFE CONSTRAINTS (monotonic violations — reliable for prefix scoring)')
print('=' * 80)
for c in sorted(prefix_safe_constraints, key=lambda x: (x.template.value, x.activities)):
    print(f'  {c.format(activity_names)}')

print(f"\n{'=' * 80}")
print('SEQUENCE CONSTRAINTS (require full trace — used as validity gate)')
print('=' * 80)
for c in sorted(sequence_constraints, key=lambda x: (x.template.value, x.activities)):
    print(f'  {c.format(activity_names)}')

print(f"\n{'=' * 80}")
print('DATA CONDITIONS')
print('=' * 80)
if data_conditions:
    for dc, cond in sorted(data_conditions.items(), key=lambda x: (x[0].template.value, x[0].activities)):
        print(f'  {dc.format(activity_names)}')
        print(f'    Condition: {cond}')
else:
    print('  (none)')

In [ ]:
# Save to pkl
import pickle

constraints_pkl = {
    'all': all_constraints,
    'prefix_safe': prefix_safe_constraints,
    'sequence': sequence_constraints,
    'data_conditions': data_conditions,
    'activity_names': activity_names,
}

pkl_path = _current / 'encoded_data' / 'domestic_declarations_constraints.pkl'

with open(pkl_path, 'wb') as f:
    pickle.dump(constraints_pkl, f)

print(f'Saved constraints to {pkl_path}')
print(f'  all: {len(constraints_pkl["all"])} constraints')
print(f'  prefix_safe: {len(constraints_pkl["prefix_safe"])} constraints')
print(f'  sequence: {len(constraints_pkl["sequence"])} constraints')
print(f'  data_conditions: {len(constraints_pkl["data_conditions"])} entries')
print(f'  activity_names: {len(constraints_pkl["activity_names"])} activities')